In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth

You shall already be familiar with the concept of association rules and the apriori algorithm. Association rule mining is a method for discovering patterns within large data sets. It focuses on identifying relationships between variables and leveraging those connections to make predictions or informed decisions. The primary objective is to uncover rules that reveal the associations between various items in the data.



### Task
Load data from data.txt file - it contains lists of grocery shopping done by nearly 2000 customers.
Store it in a boolean one hot encoded dataframe - True for items bought in a given transaction, False otherwise.

In [2]:
import ast

transactions = []
with open("data.txt", "r") as f:
    for line in f:
        # Convert string like "['milk','chicken']" into Python list
        basket = ast.literal_eval(line.strip())
        # Remove duplicates inside a single transaction
        transactions.append(list(set(basket)))

all_items = sorted({item for basket in transactions for item in basket})
df = pd.DataFrame(False, index=range(len(transactions)), columns=all_items)
for i, basket in enumerate(transactions):
    df.loc[i, basket] = True

In [3]:
df.head()

,apple,banana,beef,bread,butter,cheese,chicken,chocolate,eggs,grill,ketchup,milk,mustard,orange,pork,sausage,wagyu,yogurt
0,True,True,False,False,False,False,True,True,False,False,False,True,False,True,False,False,False,False
1,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True
2,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False
4,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False


To extract rules you can use e.g. apriori algorithm implemented in mlxtend. There are other algorithms performing the same task but using different approaches e.g. fpgrowth internally uses a tree-based structure which makes it faster in most real-life examples.

### Task
Find association rules using selected algorithm

In [4]:
from mlxtend.frequent_patterns import apriori, association_rules

freq = apriori(df, min_support=0.03, use_colnames=True)
rules = association_rules(freq, metric="lift", min_threshold=1.0)
rules.sort_values("lift", ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
110,(grill),"(sausage, mustard)",0.113257,0.060543,0.057411,0.506912,8.372795,1.0,0.050554,1.905254,0.993033,0.493274,0.475136,0.727594
107,"(sausage, mustard)",(grill),0.060543,0.113257,0.057411,0.948276,8.372795,1.0,0.050554,17.143702,0.937313,0.493274,0.941670,0.727594
96,"(ketchup, sausage)",(grill),0.052192,0.113257,0.049061,0.940000,8.299724,1.0,0.043149,14.779054,0.927945,0.421525,0.932337,0.686590
101,(grill),"(ketchup, sausage)",0.113257,0.052192,0.049061,0.433180,8.299724,1.0,0.043149,1.672149,0.991848,0.421525,0.401967,0.686590
98,"(sausage, grill)",(ketchup),0.105428,0.063152,0.049061,0.465347,7.368628,1.0,0.042403,1.752252,0.966149,0.410480,0.429306,0.621103


In [5]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

freq = fpgrowth(df, min_support=0.03, use_colnames=True)
rules = association_rules(freq, metric="confidence", min_threshold=0.3)
rules.sort_values("confidence", ascending=False).head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
55,"(grill, mustard)",(sausage),0.057411,0.248434,0.057411,1.000000,4.025210,1.0,0.043148,inf,0.797342,0.231092,1.000000,0.615546
78,"(ketchup, grill)",(sausage),0.049582,0.248434,0.049061,0.989474,3.982839,1.0,0.036743,71.398747,0.787994,0.197065,0.985994,0.593476
72,"(grill, milk)",(sausage),0.042797,0.248434,0.040710,0.951220,3.828858,1.0,0.030077,15.407098,0.771859,0.162500,0.935095,0.557543
54,"(sausage, mustard)",(grill),0.060543,0.113257,0.057411,0.948276,8.372795,1.0,0.050554,17.143702,0.937313,0.493274,0.941670,0.727594
77,"(ketchup, sausage)",(grill),0.052192,0.113257,0.049061,0.940000,8.299724,1.0,0.043149,14.779054,0.927945,0.421525,0.932337,0.686590


### Task
The association rules are characterized by high support - frequency in the dataset. Can you use this algorithm as a base and try to extract different types of rules:
 - dissociation rules e.g. buying Porshe and Rolex is not frequent in the dataset, but usually people who bought Porshe also bought Rolex
 - negative rules e.g. if someone bought low-fat milk it's unlikely there will be whole milk in the basket
 - disjunction e.g. eggs and (kielecki xor winiary ;) )
 - imagine 50% of baskets have milk and 50% of baskets have tea. If there is no relation between them then in ~25% of baskets we will have both. If milk appears together with tea in e.g. 40% of baskets it means there is a pattern. Can you find such rules and use statistical tests to check if the relation is strong?

 Send the report within 144 hours starting from the end of this class to gmiebs@cs.put.poznan.pl; start this email's subject with [IR]
